## Extract annotations

### Filter GOA data

Used the following bash commands to download and filter GOA data:

```bash
# version 2023-09-21 from GOA
wget https://ftp.ebi.ac.uk/pub/databases/GO/goa/old/UNIPROT/goa_uniprot_all.gaf.217.gz

# version 2023-12-04 from GOA
wget https://ftp.ebi.ac.uk/pub/databases/GO/goa/old/UNIPROT/goa_uniprot_all.gaf.218.gz

# Run this for each file above
pigz -dc goa_uniprot_all.gaf.218.gz | \
# pv -l | \
parallel --pipe -j$(nproc) --block 2000M \
'awk -F"\t" '\''
/^!/ { next }
{
    if ($1 == "UniProtKB" && $7 ~ /^(IDA|IPI|EXP|IGI|IMP|IEP|IC|TAS)$/ && $12 == "protein") {
        print
    }
}'\''' > filtered_goa_uniprot_exp-protein-only_218.gaf
```

### Extract annotations and label propagation

In [1]:
import pandas as pd
from pathlib import Path
from typing import Set, Tuple, Dict
import logging
from datasets import load_dataset
from cafaeval.parser import obo_parser
from cafaeval.graph import propagate
import numpy as np


# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Constants
EXPERIMENTAL_EVIDENCE_CODES = ['IDA', 'IPI', 'EXP', 'IGI', 'IMP', 'IEP', 'IC', 'TAS']
GAF_COLUMNS = [
    'DB', 'DB_ID', 'DB_Symbol', 'Qualifier', 'GO_ID', 
    'DB_Reference', 'Evidence_Code', 'With_From', 'Aspect', 
    'DB_Name', 'DB_Synonym', 'DB_Type', 'Taxon', 'Date', 
    'Assigned_By', 'Extension', 'Gene_Product_Form_ID'
]
ASPECT_NAMES = {'P': 'BP', 'F': 'MF', 'C': 'CC'}


def load_gaf(filepath: Path) -> pd.DataFrame:
    """Load GAF file, skipping header comments."""
    logger.info(f"Loading {filepath.name}")
    df = pd.read_csv(
        filepath, 
        sep='\t', 
        comment='!',
        names=GAF_COLUMNS,
        header=None,
        low_memory=False
    )
    logger.info(f"Loaded {len(df):,} annotations")
    return df


def filter_experimental_annotations(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only annotations with experimental evidence codes."""
    experimental = df[df['Evidence_Code'].isin(EXPERIMENTAL_EVIDENCE_CODES)].copy()
    logger.info(f"Filtered to {len(experimental):,} experimental annotations")
    return experimental


def create_annotation_key(df: pd.DataFrame) -> pd.Series:
    """Create unique identifier for each annotation."""
    return df['DB_ID'] + '|' + df['GO_ID'] + '|' + df['Evidence_Code']


def find_new_annotations(old_df: pd.DataFrame, new_df: pd.DataFrame, output_dir: Path = None) -> pd.DataFrame:
    """Identify annotations present in new release but not in old."""
    old_df['key'] = create_annotation_key(old_df)
    new_df['key'] = create_annotation_key(new_df)
    
    old_keys = set(old_df['key'])
    new_keys = set(new_df['key'])
    novel_keys = new_keys - old_keys
    
    logger.info(f"Found {len(novel_keys):,} new unique annotations")
    
    novel_annotations = new_df[new_df['key'].isin(novel_keys)].copy()
    
    # Analyze and save duplicates
    duplicates = novel_annotations[novel_annotations.duplicated(subset=['key'], keep=False)]
    if len(duplicates) > 0:
        logger.info(f"Found {len(duplicates):,} duplicate records for {duplicates['key'].nunique():,} annotations")
        
        # Save duplicates to file
        if output_dir:
            output_dir.mkdir(exist_ok=True)
            dup_file = output_dir / 'duplicate_annotations_analysis.tsv'
            
            # Sort by key so duplicates are grouped together
            duplicates_sorted = duplicates.sort_values(['key', 'Date'])
            duplicates_sorted.to_csv(dup_file, sep='\t', index=False)
            logger.info(f"Saved duplicate records to {dup_file}")
            
            # Also create a summary of what fields differ
            summary_file = output_dir / 'duplicate_annotations_summary.txt'
            with open(summary_file, 'w') as f:
                f.write("Duplicate Annotation Analysis\n")
                f.write("=" * 50 + "\n\n")
                f.write(f"Total duplicate records: {len(duplicates):,}\n")
                f.write(f"Unique annotations with duplicates: {duplicates['key'].nunique():,}\n\n")
                
                # Analyze differences for each duplicate group
                f.write("Sample analysis of duplicate reasons:\n\n")
                
                # Take up to 10 examples
                sample_keys = duplicates['key'].unique()[:10]
                for i, key in enumerate(sample_keys, 1):
                    group = duplicates[duplicates['key'] == key]
                    f.write(f"Example {i}: {key}\n")
                    f.write(f"  Number of duplicates: {len(group)}\n")
                    
                    # Check what fields differ
                    varying_fields = []
                    for col in ['DB_Reference', 'Date', 'Assigned_By', 'Qualifier', 'With_From']:
                        if group[col].nunique() > 1:
                            varying_fields.append(col)
                            unique_vals = group[col].unique()
                            f.write(f"  {col} varies: {unique_vals[:3]}{'...' if len(unique_vals) > 3 else ''}\n")
                    
                    if not varying_fields:
                        f.write("  All fields identical (true duplicate)\n")
                    f.write("\n")
                
                # Statistical summary
                f.write("\nDuplicate Statistics:\n")
                dup_counts = duplicates.groupby('key').size()
                f.write(f"  Average duplicates per annotation: {dup_counts.mean():.2f}\n")
                f.write(f"  Max duplicates for single annotation: {dup_counts.max()}\n")
                f.write(f"  Annotations with 2 duplicates: {(dup_counts == 2).sum()}\n")
                f.write(f"  Annotations with 3+ duplicates: {(dup_counts >= 3).sum()}\n")
                
            logger.info(f"Saved duplicate analysis to {summary_file}")
    
    novel_annotations.drop('key', axis=1, inplace=True)
    logger.info(f"Total new annotation records: {len(novel_annotations):,}")
    
    return novel_annotations


def load_cafa5_train_proteins() -> pd.DataFrame:
    """Load CAFA5 training data with GO annotations."""
    logger.info("Loading CAFA5 training data")
    ds = load_dataset("wanglab/cafa5", name="cafa5_reasoning")
    train_df = ds['train'].to_pandas()
    logger.info(f"Loaded {len(train_df):,} training proteins")
    return train_df


def load_cafa5_test_superset_proteins() -> Set[str]:
    """Load CAFA5 test superset protein IDs."""
    logger.info("Loading CAFA5 test superset")
    ds = load_dataset("wanglab/cafa5", name="cafa5_reasoning")
    test_df = ds['test_superset'].to_pandas()
    test_superset_proteins = set(test_df['protein_id'].unique())
    logger.info(f"Loaded {len(test_superset_proteins):,} test superset proteins")
    return test_superset_proteins


def filter_annotations_by_proteins(
    annotations: pd.DataFrame, 
    protein_set: Set[str]
) -> pd.DataFrame:
    """Keep only annotations for specified proteins."""
    filtered = annotations[annotations['DB_ID'].isin(protein_set)].copy()
    unique_proteins = filtered['DB_ID'].nunique()
    logger.info(f"Filtered to {len(filtered):,} annotations for {unique_proteins:,} proteins")
    return filtered


def filter_by_training_knowledge(
    test_annotations: pd.DataFrame, 
    train_proteins: pd.DataFrame,
    output_dir: Path
) -> pd.DataFrame:
    """
    Filter test annotations based on training set knowledge.
    
    Rules:
    - NK proteins (not in training): keep all annotations
    - LK proteins (in training but null in aspect): keep only annotations for null aspects
    """
    # Get unique test proteins
    test_proteins = set(test_annotations['DB_ID'].unique())
    train_protein_ids = set(train_proteins['protein_id'].unique())
    
    # Categorize proteins
    nk_proteins = test_proteins - train_protein_ids  # Not in training
    lk_candidates = test_proteins & train_protein_ids  # In both
    
    logger.info(f"Test proteins total: {len(test_proteins):,}")
    logger.info(f"NK proteins (not in training): {len(nk_proteins):,}")
    logger.info(f"Proteins in both test and training: {len(lk_candidates):,}")
    
    # For LK candidates, check which aspects are null in training
    aspect_mapping = {'P': 'go_bp', 'F': 'go_mf', 'C': 'go_cc'}
    
    # Build a dict of protein -> null aspects
    protein_null_aspects = {}
    for protein_id in lk_candidates:
        train_row = train_proteins[train_proteins['protein_id'] == protein_id].iloc[0]
        null_aspects = []
        for aspect, go_col in aspect_mapping.items():
            go_value = train_row[go_col]
            # Simple check: if it's None, it's empty
            if go_value is None:
                null_aspects.append(aspect)
        
        if null_aspects:
            protein_null_aspects[protein_id] = null_aspects
    
    lk_proteins = set(protein_null_aspects.keys())
    logger.info(f"LK proteins (in training with null aspects): {len(lk_proteins):,}")
    
    # Filter annotations
    filtered_annotations = []
    
    # Stats tracking
    stats = {
        'nk_annotations': 0,
        'lk_annotations': 0,
        'removed_annotations': 0,
        'aspect_breakdown': {'P': 0, 'F': 0, 'C': 0},
        'nk_proteins': set(),
        'lk_proteins': set(),
        'removed_proteins': set()
    }
    
    for _, ann in test_annotations.iterrows():
        protein_id = ann['DB_ID']
        aspect = ann['Aspect']
        
        if protein_id in nk_proteins:
            # NK protein - keep all annotations
            filtered_annotations.append(ann)
            stats['nk_annotations'] += 1
            stats['aspect_breakdown'][aspect] += 1
            stats['nk_proteins'].add(protein_id)
        elif protein_id in protein_null_aspects:
            # LK protein - only keep if aspect was null in training
            if aspect in protein_null_aspects[protein_id]:
                filtered_annotations.append(ann)
                stats['lk_annotations'] += 1
                stats['aspect_breakdown'][aspect] += 1
                stats['lk_proteins'].add(protein_id)
            else:
                stats['removed_annotations'] += 1
                stats['removed_proteins'].add(protein_id)
        else:
            # Protein exists in training with annotations in all aspects
            stats['removed_annotations'] += 1
            stats['removed_proteins'].add(protein_id)
    
    # Create filtered DataFrame
    filtered_df = pd.DataFrame(filtered_annotations)
    
    # Save detailed statistics
    stats_file = output_dir / 'nk_lk_filtering_stats.txt'
    with open(stats_file, 'w') as f:
        f.write("NK/LK Filtering Statistics\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Original annotations: {len(test_annotations):,}\n")
        f.write(f"Filtered annotations: {len(filtered_df):,}\n")
        f.write(f"Removed annotations: {stats['removed_annotations']:,}\n\n")
        
        f.write("Protein Categories:\n")
        f.write(f"  NK proteins: {len(nk_proteins):,}\n")
        f.write(f"  LK proteins: {len(lk_proteins):,}\n")
        f.write(f"  Fully annotated in training: {len(lk_candidates) - len(lk_proteins):,}\n\n")
        
        f.write("Annotation Breakdown:\n")
        f.write(f"  NK annotations kept: {stats['nk_annotations']:,}\n")
        f.write(f"  LK annotations kept: {stats['lk_annotations']:,}\n\n")
        
        f.write("By Aspect:\n")
        for aspect, count in stats['aspect_breakdown'].items():
            f.write(f"  {ASPECT_NAMES[aspect]}: {count:,}\n")
        
        f.write("\nUnique Proteins in Filtered Dataset:\n")
        f.write(f"  NK proteins kept: {len(stats['nk_proteins']):,}\n")
        f.write(f"  LK proteins kept: {len(stats['lk_proteins']):,}\n")
        f.write(f"  Total unique proteins: {len(stats['nk_proteins'] | stats['lk_proteins']):,}\n")
    
    logger.info(f"Saved NK/LK filtering statistics to {stats_file}")
    
    return filtered_df


def compute_statistics(annotations: pd.DataFrame) -> Dict:
    """Calculate summary statistics for annotation set."""
    stats = {
        'total_annotations': len(annotations),
        'unique_proteins': annotations['DB_ID'].nunique(),
        'aspect_counts': annotations['Aspect'].value_counts().to_dict(),
        'evidence_counts': annotations['Evidence_Code'].value_counts().to_dict(),
    }
    
    # Annotations per protein
    ann_per_protein = annotations.groupby('DB_ID').size()
    stats['annotations_per_protein'] = {
        'mean': ann_per_protein.mean(),
        'median': ann_per_protein.median(),
        'max': ann_per_protein.max(),
        'min': ann_per_protein.min()
    }
    
    return stats


def print_detailed_summary(novel_annotations: pd.DataFrame, test_superset_proteins: Set[str], 
                           current_annotations: pd.DataFrame, stage: str = ""):
    """Print detailed summary to console with stage-specific information."""
    # Calculate statistics for the CURRENT stage
    current_proteins = set(current_annotations['DB_ID'].unique())
    
    # Calculate all statistics needed
    new_annotation_proteins = set(novel_annotations['DB_ID'].unique())
    overlapping_proteins = new_annotation_proteins.intersection(test_superset_proteins)
    
    # Aspect breakdown
    aspect_counts = current_annotations['Aspect'].value_counts()
    total_annotations = len(current_annotations)
    
    # Proteins per aspect
    proteins_by_aspect = {}
    for aspect in ['P', 'F', 'C']:
        proteins_by_aspect[aspect] = set(current_annotations[current_annotations['Aspect'] == aspect]['DB_ID'].unique())
    
    # Multi-aspect proteins
    all_three = proteins_by_aspect['P'] & proteins_by_aspect['F'] & proteins_by_aspect['C']
    bp_mf = (proteins_by_aspect['P'] & proteins_by_aspect['F']) - all_three
    bp_cc = (proteins_by_aspect['P'] & proteins_by_aspect['C']) - all_three
    mf_cc = (proteins_by_aspect['F'] & proteins_by_aspect['C']) - all_three
    
    # Evidence code distribution
    evidence_counts = current_annotations['Evidence_Code'].value_counts()
    
    # Print summary
    print("\n" + "="*70)
    print(f"SUMMARY {stage}")
    print("="*70)
    
    print("\n1. PROTEIN COUNTS:")
    print(f"   - Proteins with new annotations in temporal window: {len(new_annotation_proteins):,}")
    print(f"   - Proteins in CAFA5 test superset: {len(test_superset_proteins):,}")
    print(f"   - Proteins in both (before any filtering): {len(overlapping_proteins):,}")
    
    # Stage-specific protein count
    if "NK/LK" in stage:
        print(f"   - Proteins remaining after NK/LK filtering: {len(current_proteins):,}")
        print(f"   - Reduction from original: {len(overlapping_proteins) - len(current_proteins):,} proteins removed")
    elif "PROPAGATION" in stage:
        print(f"   - Proteins after propagation: {len(current_proteins):,}")
    else:
        print(f"   - Proteins at this stage: {len(current_proteins):,}")
    
    print(f"\n2. ANNOTATIONS AT THIS STAGE:")
    print(f"   - Total annotations: {total_annotations:,}")
    if len(current_proteins) > 0:
        print(f"   - Average annotations per protein: {total_annotations/len(current_proteins):.1f}")
    
    print("\n3. BREAKDOWN BY GO ASPECT:")
    for aspect in ['P', 'F', 'C']:
        if aspect in aspect_counts:
            count = aspect_counts[aspect]
            aspect_name = ASPECT_NAMES[aspect]
            pct = count/total_annotations*100 if total_annotations > 0 else 0
            print(f"   - {aspect_name}: {count:,} annotations ({pct:.1f}%)")
    
    print("\n4. UNIQUE PROTEINS PER ASPECT:")
    for aspect in ['P', 'F', 'C']:
        aspect_name = ASPECT_NAMES[aspect]
        count = len(proteins_by_aspect.get(aspect, set()))
        print(f"   - {aspect_name}: {count:,} unique proteins")
    
    print("\n5. PROTEINS WITH ANNOTATIONS IN MULTIPLE ASPECTS:")
    print(f"   - All 3 aspects: {len(all_three):,}")
    print(f"   - BP and MF only: {len(bp_mf):,}")
    print(f"   - BP and CC only: {len(bp_cc):,}")
    print(f"   - MF and CC only: {len(mf_cc):,}")
    
    print("\n6. EVIDENCE CODE DISTRIBUTION:")
    for code in EXPERIMENTAL_EVIDENCE_CODES:
        if code in evidence_counts.index:
            count = evidence_counts[code]
            pct = count/total_annotations*100 if total_annotations > 0 else 0
            print(f"   - {code}: {count:,} ({pct:.1f}%)")
    
    print("\n" + "="*70)


def save_results(annotations: pd.DataFrame, output_dir: Path) -> None:
    """Save filtered annotations and summary statistics."""
    output_dir.mkdir(exist_ok=True)
    
    # Save annotations
    output_file = output_dir / 'cafa5_temporal_holdout_annotations.tsv'
    annotations.to_csv(output_file, sep='\t', index=False)
    logger.info(f"Saved annotations to {output_file}")
    
    # Save protein list
    proteins = sorted(annotations['DB_ID'].unique())
    protein_file = output_dir / 'cafa5_temporal_holdout_proteins.txt'
    with open(protein_file, 'w') as f:
        f.write('\n'.join(proteins))
    logger.info(f"Saved {len(proteins)} protein IDs to {protein_file}")
    
    # Save statistics
    stats = compute_statistics(annotations)
    stats_file = output_dir / 'cafa5_temporal_holdout_stats.txt'
    with open(stats_file, 'w') as f:
        f.write("CAFA5 Temporal Holdout Statistics\n")
        f.write("=" * 40 + "\n\n")
        f.write(f"Total annotations: {stats['total_annotations']:,}\n")
        f.write(f"Unique proteins: {stats['unique_proteins']:,}\n\n")
        
        f.write("Annotations by aspect:\n")
        for aspect, count in sorted(stats['aspect_counts'].items()):
            pct = count / stats['total_annotations'] * 100
            f.write(f"  {ASPECT_NAMES.get(aspect, aspect)}: {count:,} ({pct:.1f}%)\n")
        
        f.write("\nEvidence code distribution:\n")
        for code, count in sorted(stats['evidence_counts'].items(), 
                                 key=lambda x: x[1], reverse=True):
            pct = count / stats['total_annotations'] * 100
            f.write(f"  {code}: {count:,} ({pct:.1f}%)\n")
        
        f.write("\nAnnotations per protein:\n")
        for metric, value in stats['annotations_per_protein'].items():
            f.write(f"  {metric}: {value:.2f}\n")
    
    logger.info(f"Saved statistics to {stats_file}")


def propagate_annotations_cafaeval(annotations: pd.DataFrame, obo_file: Path) -> pd.DataFrame:
    """Propagate annotations using cafaeval's parser and propagation."""
    logger.info("Starting annotation propagation using cafaeval...")

    # Track all proteins from input
    all_input_proteins = set(annotations['DB_ID'].unique())
    logger.info(f"Input proteins: {len(all_input_proteins)}")
    
    # Parse OBO file using cafaeval
    ontologies = obo_parser(str(obo_file), valid_rel=("is_a", "part_of"))
    
    # Create aspect mapping
    aspect_to_namespace = {
        'P': 'biological_process',
        'F': 'molecular_function', 
        'C': 'cellular_component'
    }
    namespace_to_aspect = {v: k for k, v in aspect_to_namespace.items()}
    
    # Group annotations by aspect
    annotations_by_ns = {}
    for aspect, ns_name in aspect_to_namespace.items():
        ns_annotations = annotations[annotations['Aspect'] == aspect]
        if len(ns_annotations) > 0:
            annotations_by_ns[ns_name] = ns_annotations
    
    # Store all propagated annotations
    all_propagated = []
    
    for ns_name, ns_annotations in annotations_by_ns.items():
        if ns_name not in ontologies:
            logger.warning(f"Namespace {ns_name} not found in ontology")
            continue
            
        ont = ontologies[ns_name]
        
        # Get unique proteins in this namespace
        unique_proteins = ns_annotations['DB_ID'].unique()
        protein_to_idx = {pid: i for i, pid in enumerate(unique_proteins)}
        
        # Create matrix for propagation (proteins x GO terms)
        matrix = np.zeros((len(unique_proteins), ont.idxs), dtype='bool')
        
        # Fill matrix with original annotations
        for _, row in ns_annotations.iterrows():
            go_id = row['GO_ID']
            protein_id = row['DB_ID']
            
            if go_id in ont.terms_dict:
                protein_idx = protein_to_idx[protein_id]
                term_idx = ont.terms_dict[go_id]['index']
                matrix[protein_idx, term_idx] = 1
            elif go_id in ont.terms_dict_alt:
                # Handle alternative IDs
                for canonical_id in ont.terms_dict_alt[go_id]:
                    protein_idx = protein_to_idx[protein_id]
                    term_idx = ont.terms_dict[canonical_id]['index']
                    matrix[protein_idx, term_idx] = 1
        
        # Propagate using cafaeval
        original_matrix = matrix.copy()
        propagate(matrix, ont, ont.order, mode='max')
        
        # Extract propagated annotations
        for protein_idx, protein_id in enumerate(unique_proteins):
            # Get evidence code for this protein (assume same for all propagated)
            evidence_code = ns_annotations[ns_annotations['DB_ID'] == protein_id].iloc[0]['Evidence_Code']
            
            # Find all terms for this protein after propagation
            term_indices = np.where(matrix[protein_idx])[0]
            
            for term_idx in term_indices:
                # Find the GO ID for this index
                go_id = None
                for term_id, term_info in ont.terms_dict.items():
                    if term_info['index'] == term_idx:
                        go_id = term_id
                        break
                
                if go_id:
                    # Create annotation entry
                    # Use the original annotation as template
                    template_row = ns_annotations[ns_annotations['DB_ID'] == protein_id].iloc[0].to_dict()
                    template_row['GO_ID'] = go_id
                    template_row['Aspect'] = namespace_to_aspect[ns_name]
                    all_propagated.append(template_row)
    
    # Convert to DataFrame and remove duplicates
    propagated_df = pd.DataFrame(all_propagated)
    propagated_df = propagated_df.drop_duplicates(
        subset=['DB_ID', 'GO_ID', 'Evidence_Code'], 
        keep='first'
    )
    
    # After creating propagated_df, check for missing proteins
    propagated_proteins = set(propagated_df['DB_ID'].unique())
    missing_proteins = all_input_proteins - propagated_proteins
    
    if missing_proteins:
        logger.warning(f"Lost {len(missing_proteins)} proteins during propagation: {missing_proteins}")
        
        # Optional: Add these proteins back with empty annotations
        # This would preserve the count of 1,511
        for protein_id in missing_proteins:
            # Get original row for this protein as template
            orig_row = annotations[annotations['DB_ID'] == protein_id].iloc[0].to_dict()
    
    logger.info(f"Output proteins: {len(propagated_proteins)}")

    # Log statistics
    original_count = len(annotations)
    propagated_count = len(propagated_df)
    logger.info(f"Propagation complete:")
    logger.info(f"  Annotations: {original_count:,} -> {propagated_count:,} (+{propagated_count-original_count:,})")
    
    return propagated_df


def print_propagation_comparison(original: pd.DataFrame, propagated: pd.DataFrame):
    """Print comparison statistics between original and propagated annotations."""
    aspect_full_names = {
        'P': 'Biological Process',
        'F': 'Molecular Function',
        'C': 'Cellular Component'
    }
    
    print("\n" + "="*70)
    print("ANNOTATION PROPAGATION COMPARISON")
    print("="*70)
    
    print("\n1. OVERALL STATISTICS:")
    print(f"   Original annotations: {len(original):,}")
    print(f"   Propagated annotations: {len(propagated):,}")
    print(f"   Increase factor: {len(propagated)/len(original):.2f}x")
    
    print("\n2. UNIQUE TERMS:")
    orig_terms = set(original['GO_ID'].unique())
    prop_terms = set(propagated['GO_ID'].unique())
    new_terms = prop_terms - orig_terms
    print(f"   Original unique GO terms: {len(orig_terms):,}")
    print(f"   Propagated unique GO terms: {len(prop_terms):,}")
    print(f"   New GO terms from propagation: {len(new_terms):,}")
    
    print("\n3. BY ASPECT COMPARISON:")
    for aspect in ['P', 'F', 'C']:
        orig_count = len(original[original['Aspect'] == aspect])
        prop_count = len(propagated[propagated['Aspect'] == aspect])
        if orig_count > 0:
            increase = (prop_count - orig_count) / orig_count * 100
            print(f"   {ASPECT_NAMES[aspect]}: {orig_count:,} -> {prop_count:,} (+{increase:.1f}%)")
        else:
            print(f"   {ASPECT_NAMES[aspect]}: {orig_count:,} -> {prop_count:,}")
    
    print("\n4. GO TERMS PER PROTEIN STATISTICS (AFTER PROPAGATION):")
    
    # Get unique proteins in propagated set
    unique_proteins = propagated['DB_ID'].unique()
    
    # Overall statistics
    terms_per_protein = propagated.groupby('DB_ID')['GO_ID'].nunique()
    print(f"\n   Overall (all aspects combined):")
    print(f"     - Mean GO terms per protein: {terms_per_protein.mean():.1f}")
    print(f"     - Median GO terms per protein: {terms_per_protein.median():.0f}")
    print(f"     - Max GO terms per protein: {terms_per_protein.max()}")
    print(f"     - Min GO terms per protein: {terms_per_protein.min()}")
    
    # Per-aspect statistics
    for aspect in ['P', 'F', 'C']:
        aspect_data = propagated[propagated['Aspect'] == aspect]
        if len(aspect_data) > 0:
            aspect_terms_per_protein = aspect_data.groupby('DB_ID')['GO_ID'].nunique()
            aspect_name = ASPECT_NAMES[aspect]
            full_name = aspect_full_names[aspect]
            
            print(f"\n   {aspect_name} ({full_name}):")
            print(f"     - Proteins with {aspect_name} annotations: {len(aspect_terms_per_protein):,}")
            print(f"     - Mean GO terms per protein: {aspect_terms_per_protein.mean():.1f}")
            print(f"     - Median GO terms per protein: {aspect_terms_per_protein.median():.0f}")
            print(f"     - Max GO terms per protein: {aspect_terms_per_protein.max()}")
            print(f"     - Min GO terms per protein: {aspect_terms_per_protein.min()}")
    
    print("\n" + "="*70)


def main():
    """Main pipeline for extracting CAFA5 temporal holdout test set."""
    # File paths
    old_gaf = Path("filtered_goa_uniprot_exp-protein-only_217.gaf")
    new_gaf = Path("filtered_goa_uniprot_exp-protein-only_218.gaf")
    obo_file = Path("go-basic.obo")
    output_dir = Path("cafa5_temporal_holdout")
    
    # Load and filter GOA releases
    logger.info("Processing GOA releases")
    old_annotations = load_gaf(old_gaf)
    new_annotations = load_gaf(new_gaf)
    
    old_experimental = filter_experimental_annotations(old_annotations)
    new_experimental = filter_experimental_annotations(new_annotations)
    
    # Find new annotations
    novel_annotations = find_new_annotations(old_experimental, new_experimental, output_dir)
    
    # Filter for CAFA5 test superset proteins
    test_superset_proteins = load_cafa5_test_superset_proteins()
    test_annotations = filter_annotations_by_proteins(novel_annotations, test_superset_proteins)
    
    # Print detailed summary BEFORE NK/LK filtering
    print_detailed_summary(novel_annotations, test_superset_proteins, test_annotations, 
                          stage="BEFORE NK/LK FILTERING")
    
    # Save original annotations (before NK/LK filtering)
    original_file = output_dir / 'cafa5_temporal_holdout_annotations_original.tsv'
    test_annotations.to_csv(original_file, sep='\t', index=False)
    logger.info(f"Saved original annotations to {original_file}")
    
    # Load training data for NK/LK filtering
    logger.info("Loading CAFA5 training data for NK/LK filtering")
    train_proteins = load_cafa5_train_proteins()
    
    # Apply NK/LK filtering
    nk_lk_annotations = filter_by_training_knowledge(
        test_annotations, 
        train_proteins,
        output_dir
    )
    
    # Save NK/LK filtered annotations
    nk_lk_file = output_dir / 'cafa5_temporal_holdout_annotations_nk_lk.tsv'
    nk_lk_annotations.to_csv(nk_lk_file, sep='\t', index=False)
    logger.info(f"Saved NK/LK filtered annotations to {nk_lk_file}")
    
    # Print detailed summary AFTER NK/LK filtering but BEFORE propagation
    print_detailed_summary(novel_annotations, test_superset_proteins, nk_lk_annotations, 
                          stage="AFTER NK/LK FILTERING (BEFORE PROPAGATION)")
    
    # Load GO ontology and propagate annotations on NK/LK filtered set
    propagated_annotations = propagate_annotations_cafaeval(nk_lk_annotations, obo_file)
    
    # Print comparison between NK/LK filtered and propagated
    print_propagation_comparison(nk_lk_annotations, propagated_annotations)
    
    # Print detailed summary AFTER propagation
    print_detailed_summary(novel_annotations, test_superset_proteins, propagated_annotations, 
                          stage="AFTER NK/LK FILTERING AND PROPAGATION")
    
    # Save final propagated results
    propagated_file = output_dir / 'cafa5_temporal_holdout_annotations_propagated.tsv'
    propagated_annotations.to_csv(propagated_file, sep='\t', index=False)
    logger.info(f"Saved propagated annotations to {propagated_file}")
    
    # Save final statistics summary
    final_stats_file = output_dir / 'cafa5_temporal_holdout_final_summary.txt'
    with open(final_stats_file, 'w') as f:
        f.write("CAFA5 Temporal Holdout Pipeline Summary\n")
        f.write("=" * 50 + "\n\n")
        f.write("Files created:\n")
        f.write(f"1. {original_file.name} - All test superset annotations\n")
        f.write(f"2. {nk_lk_file.name} - NK/LK filtered annotations\n")
        f.write(f"3. {propagated_file.name} - Final propagated annotations\n\n")
        f.write("Annotation counts:\n")
        f.write(f"  Original: {len(test_annotations):,}\n")
        f.write(f"  After NK/LK filtering: {len(nk_lk_annotations):,}\n")
        f.write(f"  After propagation: {len(propagated_annotations):,}\n\n")
        f.write("Unique protein counts:\n")
        f.write(f"  Original: {test_annotations['DB_ID'].nunique():,}\n")
        f.write(f"  After NK/LK filtering: {nk_lk_annotations['DB_ID'].nunique():,}\n")
        f.write(f"  After propagation: {propagated_annotations['DB_ID'].nunique():,}\n")
    
    logger.info(f"Saved final summary to {final_stats_file}")
    print(f"\nPipeline completed successfully!")
    print(f"All results saved to: {output_dir}/")
    print(f"\nKey output files:")
    print(f"  - Original annotations: {original_file.name}")
    print(f"  - NK/LK filtered: {nk_lk_file.name}")
    print(f"  - Final propagated: {propagated_file.name}")
    print(f"  - Statistics: {output_dir}/nk_lk_filtering_stats.txt")

if __name__ == "__main__":
    main()

tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-02 17:12:07,761 - INFO - Processing GOA releases
2025-07-02 17:12:07,761 - INFO - Loading filtered_goa_uniprot_exp-protein-only_217.gaf
2025-07-02 17:12:09,559 - INFO - Loaded 1,230,139 annotations
2025-07-02 17:12:09,559 - INFO - Loading filtered_goa_uniprot_exp-protein-only_218.gaf
2025-07-02 17:12:11,327 - INFO - Loaded 1,253,198 annotations
2025-07-02 17:12:11,514 - INFO - Filtered to 1,230,139 experimental annotations
2025-07-02 17:12:11,704 - INFO - Filtered to 1,253,198 experimental annotations
2025-07-02 17:12:12,330 - INFO - Found 18,108 new unique annotations
2025-07-02 17:12:12,382 - INFO - Found 8,322 duplicate records for 3,196 annotations
2025-07-02 17:12:12,421 - INFO - Saved duplicate records to cafa5_temporal_holdout/duplicate_annotations_analysis.tsv
2025-07-


SUMMARY BEFORE NK/LK FILTERING

1. PROTEIN COUNTS:
   - Proteins with new annotations in temporal window: 6,131
   - Proteins in CAFA5 test superset: 141,864
   - Proteins in both (before any filtering): 1,511
   - Proteins remaining after NK/LK filtering: 1,511
   - Reduction from original: 0 proteins removed

2. ANNOTATIONS AT THIS STAGE:
   - Total annotations: 3,341
   - Average annotations per protein: 2.2

3. BREAKDOWN BY GO ASPECT:
   - BP: 1,776 annotations (53.2%)
   - MF: 730 annotations (21.8%)
   - CC: 835 annotations (25.0%)

4. UNIQUE PROTEINS PER ASPECT:
   - BP: 994 unique proteins
   - MF: 477 unique proteins
   - CC: 527 unique proteins

5. PROTEINS WITH ANNOTATIONS IN MULTIPLE ASPECTS:
   - All 3 aspects: 103
   - BP and MF only: 123
   - BP and CC only: 103
   - MF and CC only: 55

6. EVIDENCE CODE DISTRIBUTION:
   - IDA: 1,366 (40.9%)
   - IPI: 203 (6.1%)
   - EXP: 100 (3.0%)
   - IGI: 121 (3.6%)
   - IMP: 1,015 (30.4%)
   - IEP: 119 (3.6%)
   - IC: 40 (1.2%)
   -

2025-07-02 17:12:22,332 - INFO - Loaded 133,538 training proteins
2025-07-02 17:12:22,348 - INFO - Test proteins total: 1,511
2025-07-02 17:12:22,349 - INFO - NK proteins (not in training): 107
2025-07-02 17:12:22,349 - INFO - Proteins in both test and training: 1,404
2025-07-02 17:12:29,788 - INFO - LK proteins (in training with null aspects): 368
2025-07-02 17:12:29,843 - INFO - Saved NK/LK filtering statistics to cafa5_temporal_holdout/nk_lk_filtering_stats.txt
2025-07-02 17:12:29,848 - INFO - Saved NK/LK filtered annotations to cafa5_temporal_holdout/cafa5_temporal_holdout_annotations_nk_lk.tsv
2025-07-02 17:12:29,851 - INFO - Starting annotation propagation using cafaeval...
2025-07-02 17:12:29,851 - INFO - Input proteins: 270



SUMMARY AFTER NK/LK FILTERING (BEFORE PROPAGATION)

1. PROTEIN COUNTS:
   - Proteins with new annotations in temporal window: 6,131
   - Proteins in CAFA5 test superset: 141,864
   - Proteins in both (before any filtering): 1,511
   - Proteins remaining after NK/LK filtering: 270
   - Reduction from original: 1,241 proteins removed

2. ANNOTATIONS AT THIS STAGE:
   - Total annotations: 585
   - Average annotations per protein: 2.2

3. BREAKDOWN BY GO ASPECT:
   - BP: 226 annotations (38.6%)
   - MF: 195 annotations (33.3%)
   - CC: 164 annotations (28.0%)

4. UNIQUE PROTEINS PER ASPECT:
   - BP: 123 unique proteins
   - MF: 123 unique proteins
   - CC: 108 unique proteins

5. PROTEINS WITH ANNOTATIONS IN MULTIPLE ASPECTS:
   - All 3 aspects: 18
   - BP and MF only: 22
   - BP and CC only: 13
   - MF and CC only: 13

6. EVIDENCE CODE DISTRIBUTION:
   - IDA: 224 (38.3%)
   - IPI: 71 (12.1%)
   - EXP: 16 (2.7%)
   - IGI: 11 (1.9%)
   - IMP: 150 (25.6%)
   - IEP: 18 (3.1%)
   - IC: 17 (2.

2025-07-02 17:12:31,610 - INFO - Ontology: biological_process, total 27941, roots 1, leaves 13286, alternative_ids 2159
2025-07-02 17:12:31,897 - INFO - Ontology: molecular_function, total 11262, roots 1, leaves 9215, alternative_ids 1090
2025-07-02 17:12:31,941 - INFO - Ontology: cellular_component, total 4042, roots 1, leaves 2751, alternative_ids 242
2025-07-02 17:12:38,414 - WARNING - Lost 6 proteins during propagation: {'Q9VKA5', 'Q9VKJ7', 'Q5K6N0', 'Q6AZ50', 'Q9VSH2', 'O94730'}
2025-07-02 17:12:38,415 - INFO - Output proteins: 264
2025-07-02 17:12:38,416 - INFO - Propagation complete:
2025-07-02 17:12:38,416 - INFO -   Annotations: 585 -> 4,846 (+4,261)
2025-07-02 17:12:38,474 - INFO - Saved propagated annotations to cafa5_temporal_holdout/cafa5_temporal_holdout_annotations_propagated.tsv
2025-07-02 17:12:38,476 - INFO - Saved final summary to cafa5_temporal_holdout/cafa5_temporal_holdout_final_summary.txt



ANNOTATION PROPAGATION COMPARISON

1. OVERALL STATISTICS:
   Original annotations: 585
   Propagated annotations: 4,846
   Increase factor: 8.28x

2. UNIQUE TERMS:
   Original unique GO terms: 299
   Propagated unique GO terms: 1,338
   New GO terms from propagation: 1,045

3. BY ASPECT COMPARISON:
   BP: 226 -> 2,886 (+1177.0%)
   MF: 195 -> 789 (+304.6%)
   CC: 164 -> 1,171 (+614.0%)

4. GO TERMS PER PROTEIN STATISTICS (AFTER PROPAGATION):

   Overall (all aspects combined):
     - Mean GO terms per protein: 18.4
     - Median GO terms per protein: 13
     - Max GO terms per protein: 100
     - Min GO terms per protein: 3

   BP (Biological Process):
     - Proteins with BP annotations: 120
     - Mean GO terms per protein: 24.1
     - Median GO terms per protein: 23
     - Max GO terms per protein: 91
     - Min GO terms per protein: 3

   MF (Molecular Function):
     - Proteins with MF annotations: 118
     - Mean GO terms per protein: 6.7
     - Median GO terms per protein: 5
  

## Prepare extracted test set for pushing to HF

### Load extracted data from output files

In [2]:
def load_temporal_holdout_to_dataset_format(annotations_file: Path) -> pd.DataFrame:
    """
    Load temporal holdout annotations and convert to CAFA5 dataset format.
    
    Args:
        annotations_file: Path to the annotations TSV file
    
    Returns:
        DataFrame with columns matching CAFA5 dataset format
    """
    # Load annotations
    annotations = pd.read_csv(annotations_file, sep='\t')
    
    # Group by protein and aggregate GO terms by aspect
    protein_data = []
    
    for protein_id, group in annotations.groupby('DB_ID'):
        # Separate GO IDs by aspect
        bp_terms = group[group['Aspect'] == 'P']['GO_ID'].unique().tolist()
        mf_terms = group[group['Aspect'] == 'F']['GO_ID'].unique().tolist()
        cc_terms = group[group['Aspect'] == 'C']['GO_ID'].unique().tolist()
        
        # Convert empty lists to None to match training data format
        bp_terms = bp_terms if bp_terms else None
        mf_terms = mf_terms if mf_terms else None
        cc_terms = cc_terms if cc_terms else None
        
        # All GO IDs combined (only include non-None lists)
        all_go_ids = []
        if bp_terms:
            all_go_ids.extend(bp_terms)
        if mf_terms:
            all_go_ids.extend(mf_terms)
        if cc_terms:
            all_go_ids.extend(cc_terms)
        
        # If no GO IDs at all, set to None
        all_go_ids = all_go_ids if all_go_ids else None
        
        # Create row in target format
        row = {
            'protein_id': protein_id,
            'protein_names': None,  # Not available in GAF
            'protein_function': None,  # Not available in GAF
            'organism': None,  # Could extract from Taxon field if needed
            'length': None,  # Not available in GAF
            'subcellular_location': None,  # Not available in GAF
            'sequence': None,  # Not available in GAF
            'go_ids': all_go_ids,
            'go_bp': bp_terms,
            'go_mf': mf_terms,
            'go_cc': cc_terms,
            'interpro_ids': None,  # Not available in GAF
            'structure_path': None,  # Not available in GAF
            'string_id': None,  # Not available in GAF
            'interaction_partners': None,  # Not available in GAF
            'full_interaction_info': None  # Not available in GAF
        }
        
        protein_data.append(row)
    
    # Convert to DataFrame
    df = pd.DataFrame(protein_data)
    
    # Sort by protein_id for consistency
    df = df.sort_values('protein_id').reset_index(drop=True)
    
    print(f"Loaded {len(df)} proteins with temporal holdout annotations")
    print(f"Average GO terms per protein:")
    print(f"  - Total: {df['go_ids'].apply(lambda x: len(x) if x else 0).mean():.1f}")
    print(f"  - BP: {df['go_bp'].apply(lambda x: len(x) if x else 0).mean():.1f}")
    print(f"  - MF: {df['go_mf'].apply(lambda x: len(x) if x else 0).mean():.1f}")
    print(f"  - CC: {df['go_cc'].apply(lambda x: len(x) if x else 0).mean():.1f}")
    
    return df
    

# Load propagated annotations
df_temporal = load_temporal_holdout_to_dataset_format(
    Path("cafa5_temporal_holdout/cafa5_temporal_holdout_annotations_propagated.tsv")
)

Loaded 264 proteins with temporal holdout annotations
Average GO terms per protein:
  - Total: 18.4
  - BP: 10.9
  - MF: 3.0
  - CC: 4.4


### Merge extracted GO terms with the already existing test set protein data

In [3]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path

def merge_temporal_holdout_with_hf_test_superset(annotations_file: Path) -> pd.DataFrame:
    """
    Merge temporal holdout GO annotations with HuggingFace test superset dataset information.
    
    Args:
        annotations_file: Path to the temporal holdout annotations TSV file
    
    Returns:
        DataFrame with GO annotations and all protein information from HF test superset
    """
    # Load HuggingFace test superset dataset
    print("Loading HuggingFace test_superset dataset...")
    test_superset_dataset = load_dataset("wanglab/cafa5", "cafa5_reasoning", split="test_superset")
    test_superset_df = test_superset_dataset.to_pandas()
    
    # Load temporal holdout annotations
    print(f"Loading temporal holdout annotations from {annotations_file}...")
    df_temporal = load_temporal_holdout_to_dataset_format(annotations_file)
    
    # Get protein IDs that have temporal holdout annotations
    temporal_proteins = set(df_temporal['protein_id'])
    print(f"Found {len(temporal_proteins)} proteins with temporal holdout annotations")
    
    # Filter HF test superset to only these proteins
    test_superset_subset = test_superset_df[test_superset_df['protein_id'].isin(temporal_proteins)].copy()
    print(f"Found {len(test_superset_subset)} matching proteins in HF test superset")
    
    # Drop GO columns from HF dataset (we'll use our temporal holdout GO terms)
    go_columns = ['go_ids', 'go_bp', 'go_mf', 'go_cc']
    test_superset_subset = test_superset_subset.drop(columns=go_columns)
    
    # Merge with temporal holdout GO annotations
    merged_df = test_superset_subset.merge(
        df_temporal[['protein_id'] + go_columns],
        on='protein_id',
        how='inner'
    )
    
    print(f"\nMerge complete:")
    print(f"- Total proteins with annotations: {len(merged_df)}")
    print(f"- Proteins with sequences: {merged_df['sequence'].notna().sum()}")
    print(f"- Proteins with organism info: {merged_df['organism'].notna().sum()}")
    print(f"- Proteins with InterPro IDs: {merged_df['interpro_ids'].notna().sum()}")
    
    # Verify all expected columns are present
    expected_columns = [
        'protein_id', 'protein_names', 'protein_function', 'organism', 
        'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 
        'go_mf', 'go_cc', 'interpro_ids', 'structure_path', 'string_id', 
        'interaction_partners', 'full_interaction_info'
    ]
    
    # Ensure column order matches expected format
    merged_df = merged_df[expected_columns]
    
    return merged_df


# Merge propagated annotations with HF test superset data to create the actual test set
df_test = merge_temporal_holdout_with_hf_test_superset(
    Path("cafa5_temporal_holdout/cafa5_temporal_holdout_annotations_propagated.tsv")
)
df_test

Loading HuggingFace test_superset dataset...
Loading temporal holdout annotations from cafa5_temporal_holdout/cafa5_temporal_holdout_annotations_propagated.tsv...
Loaded 264 proteins with temporal holdout annotations
Average GO terms per protein:
  - Total: 18.4
  - BP: 10.9
  - MF: 3.0
  - CC: 4.4
Found 264 proteins with temporal holdout annotations
Found 264 matching proteins in HF test superset

Merge complete:
- Total proteins with annotations: 264
- Proteins with sequences: 264
- Proteins with organism info: 264
- Proteins with InterPro IDs: 262


,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,interpro_ids,structure_path,string_id,interaction_partners,full_interaction_info
0,Q80TR1,Adhesion G protein-coupled receptor L1 (Calciu...,Calcium-independent receptor of high affinity ...,Mus musculus (Mouse),1466.0,Cell membrane; Multi-pass membrane protein. Ce...,MARLAAALWSLCVTTVLVTSATQGLSRAGLPFGLMRRELACEGYPI...,"[GO:0003674, GO:0005488, GO:0005515]",None,"[GO:0003674, GO:0005488, GO:0005515]",None,"[IPR032471, IPR046338, IPR017981, IPR036445, I...",AF-Q80TR1-F1-model_v4.cif,10090.ENSMUSP00000118452,"[Gnb4, Frmd6, Unc5a, Gnb5, Tenm1, Flrt2, Unc5c...","[{'coexpression': 95, 'combined_score': 814, '..."
1,Q6GQW0,Ankyrin repeat- and BTB/POZ domain-containing ...,None,Mus musculus (Mouse),1109.0,Membrane {ECO:0000305}; Single-pass membrane p...,MARRGKKPVVRTLEDLTLDSGYGGAADSVRSSNLSLCCSDSHPASP...,"[GO:0003674, GO:0005488, GO:0005515, GO:001990...",None,"[GO:0003674, GO:0005488, GO:0005515, GO:001990...","[GO:0005575, GO:0030054, GO:0045202, GO:009897...","[IPR052089, IPR002110, IPR036770, IPR000210, I...",AF-Q6GQW0-F1-model_v4.cif,10090.ENSMUSP00000100944,None,None
2,A2AU72,Armadillo repeat-containing protein 3,None,Mus musculus (Mouse),881.0,None,MGKKIKKEVEPPPKDVFDPITIESKKAATVVLMLKSPEEDILAKAC...,"[GO:0000003, GO:0001539, GO:0003006, GO:000334...","[GO:0000003, GO:0001539, GO:0003006, GO:000334...","[GO:0003674, GO:0005488, GO:0005515]",None,"[IPR011989, IPR016024, IPR000225, IPR052441, I...",AF-A2AU72-F1-model_v4.cif,10090.ENSMUSP00000110287,"[Iqub, Efcab1, 1700007K13Rik, Tmem95, Ccdc189,...","[{'coexpression': 111, 'combined_score': 752, ..."
3,Q3V079,Basal body-orientation factor 1 (Coiled-coil d...,Basal body protein required in multiciliate ce...,Mus musculus (Mouse),533.0,"Cytoplasm, cytoskeleton, cilium basal body {EC...",MPAKDKRKDKRKDKRKGKNKGKEPKKIIKSDEPAIERAKANASLWE...,"[GO:0000003, GO:0000226, GO:0001539, GO:000157...","[GO:0000003, GO:0000226, GO:0001539, GO:000157...","[GO:0003674, GO:0005488, GO:0005515]","[GO:0005575, GO:0005622, GO:0005737, GO:000585...",[IPR032777],AF-Q3V079-F1-model_v4.cif,10090.ENSMUSP00000080512,[Ccdc78],"[{'coexpression': 0, 'combined_score': 726, 'c..."
4,O70552,Protein BTG4 (BTG family member 4) (Protein PC3b),"Adapter protein that bridges CNOT7, a catalyti...",Mus musculus (Mouse),250.0,None,MRDEIATAVFFVTRLVKKHEKLSTQQIETFALKLMTILFEKYRGHW...,"[GO:0005575, GO:0005622, GO:0005737, GO:000582...",None,None,"[GO:0005575, GO:0005622, GO:0005737, GO:000582...","[IPR002087, IPR033332, IPR036054]",AF-O70552-F1-model_v4.cif,10090.ENSMUSP00000110074,"[Cnot7, Cnot8, Pabpn1l, Cnot6l]","[{'coexpression': 0, 'combined_score': 952, 'c..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,O13398,Sodium/potassium exporting P-type ATPase 2 (EC...,None,Schwanniomyces occidentalis (Yeast) (Debaryomy...,1082.0,Cell membrane {ECO:0000305|PubMed:9430707}; Mu...,MSSINTNVAEKHSYKERVNTDASSLSADGSVEAHRLTIEQVAKIFN...,"[GO:0003674, GO:0005215, GO:0008324, GO:000855...",None,"[GO:0003674, GO:0005215, GO:0008324, GO:000855...",None,"[IPR006068, IPR004014, IPR023299, IPR018303, I...",AF-O13398-F1-model_v4.cif,None,None,None
260,C1L360,Sodium/potassium exporting P-type ATPase 1 (EC...,None,Marchantia polymorpha (Common liverwort) (Marc...,951.0,Cell membrane {ECO:0000250|UniProtKB:Q7XB51}; ...,MMGANSTEWHGQSVEQVTELLGTDVERGLKESVVGQLQKQFGPNEL...,"[GO:0003674, GO:0005215, GO:0008324, GO:000855...",None,"[GO:0003674, GO:0005215, GO:0008324, GO:000855...",None,"[IPR006068, IPR004014, IPR023299, IPR018303, I...",AF-C1L360-F1-model_v4.cif,None,None,None
261,A0A7M7H308,Waprin-Thr1,None,Apis mellifera (Honeybee),110.0,Secreted {ECO:0000269|PubMed:36738900}.,MYKKGTILVLAYLLIATAVCQLSYKEGHCPLRNSVSKCIPRCVSDY...,"[GO:0008150, GO:0044278, GO:0044419, GO:0140975]","[GO:0008150, GO:0044278, GO:0044419, GO:0140975]",None,None,"[IPR036645, IPR008197]",AF-A0A7M7H308-F1-model_v4.cif,None,None,None
262,Q64368,None,"R

#### Tests: Quick check

In [4]:
test_superset = load_dataset("wanglab/cafa5", "cafa5_reasoning", split="test_superset")
test_superset_df = test_superset.to_pandas()

test_superset_df[test_superset_df['protein_id'] == 'Q9D8X5']

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,interpro_ids,structure_path,string_id,interaction_partners,full_interaction_info
2801,Q9D8X5,CCR4-NOT transcription complex subunit 8 (EC 3...,Has 3'-5' poly(A) exoribonuclease activity for...,Mus musculus (Mouse),292.0,Cytoplasm {ECO:0000250}. Nucleus {ECO:0000250}.,MPAALVENSQVICEVWASNLEEEMRKIREIVLSYSYIAMDTEFPGV...,[None],[None],[None],[None],"[IPR039637, IPR006941, IPR012337, IPR036397]",AF-Q9D8X5-F1-model_v4.cif,10090.ENSMUSP00000104471,"[Cnot6l, Cnot10, Cnot4, Tob2, Cnot9, Noct, Kif...","[{'coexpression': 52, 'combined_score': 998, '..."


In [5]:
df_test[df_test['protein_id'] == 'Q9D8X5']

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,interpro_ids,structure_path,string_id,interaction_partners,full_interaction_info
17,Q9D8X5,CCR4-NOT transcription complex subunit 8 (EC 3...,Has 3'-5' poly(A) exoribonuclease activity for...,Mus musculus (Mouse),292.0,Cytoplasm {ECO:0000250}. Nucleus {ECO:0000250}.,MPAALVENSQVICEVWASNLEEEMRKIREIVLSYSYIAMDTEFPGV...,"[GO:0005575, GO:0005622, GO:0005737, GO:000582...",None,None,"[GO:0005575, GO:0005622, GO:0005737, GO:000582...","[IPR039637, IPR006941, IPR012337, IPR036397]",AF-Q9D8X5-F1-model_v4.cif,10090.ENSMUSP00000104471,"[Cnot6l, Cnot10, Cnot4, Tob2, Cnot9, Noct, Kif...","[{'coexpression': 52, 'combined_score': 998, '..."


In [6]:
[i for i in df_test[df_test['protein_id'] == 'Q9D8X5'].go_ids]

[['GO:0005575', 'GO:0005622', 'GO:0005737', 'GO:0005829', 'GO:0110165']]

#### Tests: Compare LK proteins from train and test

In [7]:
import random

# Load the NK/LK annotations to get the protein lists
nk_lk_df = pd.read_csv('cafa5_temporal_holdout/cafa5_temporal_holdout_annotations_nk_lk.tsv', sep='\t')

# Load training data
train_dataset = load_dataset("wanglab/cafa5", "cafa5_reasoning", split="train")
train_df = train_dataset.to_pandas()

# Get LK proteins (proteins that exist in both)
train_protein_ids = set(train_df['protein_id'])
nk_lk_proteins = set(nk_lk_df['DB_ID'].unique())
lk_proteins = nk_lk_proteins & train_protein_ids

# Sample LK proteins
n_samples = 20
lk_sample = random.sample(list(lk_proteins), min(n_samples, len(lk_proteins)))

In [8]:
train_df[train_df['protein_id'].isin(lk_sample)][['protein_id', 'go_bp', 'go_mf', 'go_cc']].sort_values('protein_id')

,protein_id,go_bp,go_mf,go_cc
2202,A0A1L8G2K9,"[GO:0008150, GO:0008152, GO:0009987, GO:005089...",None,None
95692,O43037,None,None,"[GO:0005575, GO:0110165, GO:0005829, GO:000562..."
23898,P10518,"[GO:0008150, GO:0008152, GO:0050896, GO:000998...","[GO:0003674, GO:0003824, GO:0016829, GO:001683...",None
126496,Q2T9T3,None,"[GO:0003674, GO:0005488, GO:0005515]",None
109413,Q5T3F8,None,None,"[GO:0005575, GO:0110165, GO:0005622, GO:004322..."
109937,Q64565,None,None,"[GO:0005575, GO:0110165, GO:0005622, GO:004322..."
53921,Q6GQW0,"[GO:0008150, GO:0050896, GO:0050789, GO:000998...",None,None
110593,Q6LA53,None,None,"[GO:0005575, GO:0110165, GO:0005829, GO:000562..."
110449,Q6NX65,None,None,"[GO:0005575, GO:0110165, GO:0005622, GO:004322..."
56140,Q71UE8,"[GO:0008150, GO:0008152, GO:0050896, GO:004222...",None,"[GO:0005575, GO:0110165, GO:0005622, GO:004322..."


In [9]:
df_test[df_test['protein_id'].isin(lk_sample)][['protein_id', 'go_bp', 'go_mf', 'go_cc']].sort_values('protein_id')

,protein_id,go_bp,go_mf,go_cc
166,A0A1L8G2K9,None,"[GO:0003674, GO:0003824, GO:0004175, GO:000422...",None
105,O43037,"[GO:0000375, GO:0000377, GO:0000398, GO:000613...",None,None
33,P10518,None,None,"[GO:0005575, GO:0005622, GO:0005737, GO:000582..."
251,Q2T9T3,None,None,"[GO:0000428, GO:0005575, GO:0005622, GO:000563..."
182,Q5T3F8,None,"[GO:0003674, GO:0005215, GO:0005216, GO:000838...",None
60,Q64565,"[GO:0006082, GO:0006520, GO:0006575, GO:000680...",None,None
1,Q6GQW0,None,"[GO:0003674, GO:0005488, GO:0005515, GO:001990...","[GO:0005575, GO:0030054, GO:0045202, GO:009897..."
96,Q6LA53,"[GO:0000375, GO:0000377, GO:0000398, GO:000613...",None,None
74,Q6NX65,"[GO:0000003, GO:0003006, GO:0003008, GO:000695...",None,None
71,Q71UE8,None,"[GO:0003674, GO:0005488, GO:0005515]",None


#### Test: Check dropped proteins

In [ ]:
import random

# Load the ORIGINAL annotations before NK/LK filtering
original_df = pd.read_csv('cafa5_temporal_holdout/cafa5_temporal_holdout_annotations_original.tsv', sep='\t')

# Load the NK/LK filtered annotations
nk_lk_df = pd.read_csv('cafa5_temporal_holdout/cafa5_temporal_holdout_annotations_nk_lk.tsv', sep='\t')

# Load training data
train_dataset = load_dataset("wanglab/cafa5", "cafa5_reasoning", split="train")
train_df = train_dataset.to_pandas()

In [ ]:
# Get proteins that were completely dropped
original_proteins = set(original_df['DB_ID'].unique())
nk_lk_proteins = set(nk_lk_df['DB_ID'].unique())
train_protein_ids = set(train_df['protein_id'].unique())

# Dropped proteins = in original AND in training BUT NOT in nk_lk
dropped_proteins = (original_proteins & train_protein_ids) - nk_lk_proteins

print(f"Total proteins completely dropped: {len(dropped_proteins)}")

# Sample some dropped proteins
n_samples = 15
dropped_sample = random.sample(list(dropped_proteins), min(n_samples, len(dropped_proteins)))

# Show training data (these proteins had NO new annotations for their null aspects)
print("\nTRAINING DATA for dropped proteins:")
print("(These proteins either had all aspects filled OR received no new annotations for their null aspects)")
train_dropped = train_df[train_df['protein_id'].isin(dropped_sample)][['protein_id', 'go_bp', 'go_mf', 'go_cc']].sort_values('protein_id')
train_dropped

Total proteins completely dropped: 1241

TRAINING DATA for dropped proteins:
(These proteins either had all aspects filled OR received no new annotations for their null aspects)


,protein_id,go_bp,go_mf,go_cc
14376,O00370,"[GO:0008150, GO:0008152, GO:0009987, GO:004423...","[GO:0003674, GO:0003824, GO:0016740, GO:001678...",None
15608,O14261,"[GO:0008150, GO:0008152, GO:0009987, GO:005089...","[GO:0003674, GO:0098772, GO:0005488, GO:003023...","[GO:0005575, GO:0032991, GO:0110165, GO:000562..."
18262,O75503,"[GO:0008150, GO:0008152, GO:0065007, GO:005117...","[GO:0003674, GO:0005488, GO:0036094, GO:003024...","[GO:0005575, GO:0110165, GO:0005622, GO:004322..."
21021,P04689,"[GO:0008150, GO:0051179, GO:0009987, GO:005164...","[GO:0003674, GO:0005198, GO:0005488, GO:000520...","[GO:0005575, GO:0110165, GO:0005622, GO:004322..."
26270,P22413,"[GO:0008150, GO:0051179, GO:0050789, GO:003250...","[GO:0003674, GO:0003824, GO:0005488, GO:003609...","[GO:0005575, GO:0110165, GO:0005622, GO:004322..."
26998,P24393,"[GO:0008150, GO:0065007, GO:0050896, GO:005078...",None,"[GO:0005575, GO:0110165, GO:0005622, GO:004322..."
38458,P9WMH7,"[GO:0008150, GO:0048518, GO:0050789, GO:004441...","[GO:0003674, GO:0005488, GO:0097159, GO:190136...",None
39786,Q06132,"[GO:0008150, GO:0008152, GO:0009987, GO:005089...","[GO:0003674, GO:0005488, GO:0005515]","[GO:0005575, GO:0032991, GO:0110165, GO:199090..."
42189,Q13496,"[GO:0008150, GO:0008152, GO:0051179, GO:005078...","[GO:0003674, GO:0005488, GO:0003824, GO:001678...","[GO:0005575, GO:0110165, GO:0031252, GO:000562..."
53206,Q64323,"[GO:0008150, GO:0008152, GO:0050896, GO:000998...",None,None


In [ ]:
# Analyze WHY each protein was dropped
print("ANALYSIS: Why were these proteins dropped?")
print("="*70)

for _, train_row in train_dropped.iterrows():
    protein_id = train_row['protein_id']
    print(f"\nProtein {protein_id}:")
    
    # Check which aspects were null and non-null in training
    null_aspects = []
    non_null_aspects = []
    
    if train_row['go_bp'] is None:
        null_aspects.append('P')
    else:
        non_null_aspects.append('P')
        
    if train_row['go_mf'] is None:
        null_aspects.append('F')
    else:
        non_null_aspects.append('F')
        
    if train_row['go_cc'] is None:
        null_aspects.append('C')
    else:
        non_null_aspects.append('C')
    
    print(f"  Non-null aspects in training: {non_null_aspects if non_null_aspects else 'NONE'}")
    print(f"  Null aspects in training: {null_aspects if null_aspects else 'NONE'}")
    
    # Get new annotations that were attempted
    new_anns = original_df[original_df['DB_ID'] == protein_id]
    new_aspects = new_anns['Aspect'].unique()
    print(f"  New annotation aspects: {list(new_aspects)}")
    
    # Explain why dropped
    if not null_aspects:
        print(f"  → Dropped because: All aspects already had annotations in training")
    else:
        overlapping_aspects = [asp for asp in new_aspects if asp in non_null_aspects]
        if overlapping_aspects:
            print(f"  → Dropped because: New annotations were only for already-filled aspects {overlapping_aspects}")

ANALYSIS: Why were these proteins dropped?

Protein O00370:
  Non-null aspects in training: ['P', 'F']
  Null aspects in training: ['C']
  New annotation aspects: ['P']
  → Dropped because: New annotations were only for already-filled aspects ['P']

Protein O14261:
  Non-null aspects in training: ['P', 'F', 'C']
  Null aspects in training: NONE
  New annotation aspects: ['P']
  → Dropped because: All aspects already had annotations in training

Protein O75503:
  Non-null aspects in training: ['P', 'F', 'C']
  Null aspects in training: NONE
  New annotation aspects: ['F']
  → Dropped because: All aspects already had annotations in training

Protein P04689:
  Non-null aspects in training: ['P', 'F', 'C']
  Null aspects in training: NONE
  New annotation aspects: ['P']
  → Dropped because: All aspects already had annotations in training

Protein P22413:
  Non-null aspects in training: ['P', 'F', 'C']
  Null aspects in training: NONE
  New annotation aspects: ['P']
  → Dropped because: All

### Create HF dataset and push to HF

In [10]:
from datasets import Dataset

test_hf = Dataset.from_pandas(df_test, preserve_index=False)
test_hf

Dataset({
    features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path', 'string_id', 'interaction_partners', 'full_interaction_info'],
    num_rows: 264
})

In [ ]:
# test_hf.push_to_hub(
#     repo_id="wanglab/cafa5",
#     config_name="cafa5_reasoning",
#     split="test",
#     commit_message="Filter test set to only include NK/LK proteins",
# )

Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]


CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/44c8d1894605a0ec3b29604b5f633f0a0b827769', commit_message='Filter test set to only include NK/LK proteins', commit_description='', oid='44c8d1894605a0ec3b29604b5f633f0a0b827769', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)